# FunctionGemma Fine-Tuning (Colab)

Fine-tune `google/functiongemma-270m-it` to call Cardinal System tools.

**Requirements:** Runtime -> Change runtime type -> T4 GPU

**Steps:**
1. Run Cell 1 (install deps + login)
2. Run Cell 2 (upload training_data.jsonl)
3. Run Cell 3 (train)
4. Run Cell 4 (push to Hugging Face)

In [ ]:
!pip install -q trl transformers accelerate huggingface-hub

# Disable Xet Storage which causes tokenizer.json to hang at 25%
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import login
login()

In [ ]:
from google.colab import files
import zipfile, os

print("Upload training_data.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Loaded: {DATA_FILE}")

print("\nUpload functiongemma-270m-it.zip (model files):")
uploaded = files.upload()
zip_name = [k for k in uploaded.keys() if k.endswith('.zip')]
if zip_name:
    os.makedirs("model_local", exist_ok=True)
    with zipfile.ZipFile(zip_name[0], "r") as zf:
        zf.extractall("model_local")
    print(f"Extracted to model_local/ ({len(os.listdir('model_local'))} files)")
    MODEL_PATH = "model_local"
else:
    print("No zip uploaded, will use Hugging Face")
    MODEL_PATH = None

In [ ]:
import json, torch, os
os.environ["HF_HUB_DISABLE_XET"] = "1"
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

HF_TOKEN = "YOUR_HF_TOKEN"  # Replace with your HF token from Cell 1
OUTPUT_DIR = "functiongemma-finetuned"

# Use local model if zip uploaded, else download via CLI (avoids Python HTTP freeze)
if MODEL_PATH is None or not os.path.exists(MODEL_PATH):
    MODEL = "google/functiongemma-270m-it"
    !huggingface-cli download {MODEL} --token {HF_TOKEN}
else:
    MODEL = MODEL_PATH
    print(f"Using local model from {MODEL}")

# Load training data
records = []
with open(DATA_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"Loaded {len(records)} samples")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Format into prompt-completion pairs
def format_pc(sample):
    msgs = sample["messages"]
    tools = sample.get("tools")
    full = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=False)
    prompt = tokenizer.apply_chat_template(msgs[:-1], tools=tools, tokenize=False, add_generation_prompt=True)
    completion = full[len(prompt):]
    return {"prompt": prompt, "completion": completion}

pc_records = [format_pc(r) for r in records]

split_idx = int(len(pc_records) * 0.9)
train_ds = Dataset.from_dict({"prompt": [r["prompt"] for r in pc_records[:split_idx]],
                              "completion": [r["completion"] for r in pc_records[:split_idx]]})
val_ds = Dataset.from_dict({"prompt": [r["prompt"] for r in pc_records[split_idx:]],
                            "completion": [r["completion"] for r in pc_records[split_idx:]]})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"Prompt preview: {pc_records[0]['prompt'][:200]}...")
print(f"Completion: {pc_records[0]['completion'][:200]}...")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="eager", token=HF_TOKEN
)
print("Model loaded on:", model.device)

# Training config (TRL 1.8.0: max_seq_length and tokenizer go in SFTTrainer, not SFTConfig)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    bf16=True,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    max_seq_length=2048,
)

print("Starting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(
    repo_id="skgufranahamed/functiongemma-finetuned",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="skgufranahamed/functiongemma-finetuned",
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/skgufranahamed/functiongemma-finetuned")